# 🏦 Enterprise Mock Data Loader

**Generates 3 years of realistic Databricks cost & usage data** modeled after large financial institutions (enterprise scale):

| Metric | Value |
|--------|-------|
| Annual spend forecast | ~$5M |
| Active users | 1,100+ |
| Workspaces | 5 LOBs |
| Clusters | 200 |
| SQL Warehouses | 30 |
| Jobs | 500 |
| ML Experiments | 20 |
| DLT Pipelines | 20 |
| Serving Endpoints | 8 |

**Run all cells below** — takes ~15-30 min depending on warehouse size.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# If running in Databricks notebook, these are auto-injected.
# If running locally, set environment variables.

import os, sys

# Try Databricks notebook context first
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    HOST = ctx.apiUrl().get()
    TOKEN = ctx.apiToken().get()
    print(f'Running in Databricks notebook: {HOST}')
except:
    HOST = os.environ.get('DATABRICKS_HOST', '')
    TOKEN = os.environ.get('DATABRICKS_TOKEN', '')
    print(f'Running locally: {HOST}')

WAREHOUSE_ID = os.environ.get('DATABRICKS_WAREHOUSE_ID', '21f5bd20b7f44a51')
print(f'Warehouse: {WAREHOUSE_ID}')

In [ ]:
# ── SQL Execution Helper ──────────────────────────────────────────────
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

client = WorkspaceClient(host=HOST, token=TOKEN)

def run_sql(label, sql, timeout_sec=300):
    """Execute SQL and wait for completion."""
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    deadline = time.time() + timeout_sec
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline:
            print(f'  ⏱ TIMEOUT: {label}')
            return False
        time.sleep(3)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED:
        print(f'  ✅ {label}')
        return True
    err = getattr(resp.status.error, 'message', str(resp.status.state))
    print(f'  ⚠️ SKIP ({err[:100]}): {label}')
    return False

print('SQL executor ready')

In [ ]:
# ── Constants & Data Generation ───────────────────────────────────────
import random
from datetime import datetime, timedelta

ACCOUNT_ID = 'acc-corp-001'
WORKSPACES = [
    ('ws-100001', 'prod-consumer-banking', 'us-east-1'),
    ('ws-100002', 'prod-investment-banking', 'us-east-1'),
    ('ws-100003', 'prod-risk-analytics', 'us-west-2'),
    ('ws-100004', 'prod-data-platform', 'us-east-1'),
    ('ws-100005', 'prod-fraud-detection', 'us-west-2'),
]

DEPARTMENTS = [
    'Consumer Banking', 'Investment Banking', 'Risk Analytics',
    'Fraud Detection', 'Data Engineering', 'Data Science',
    'Compliance', 'Treasury', 'Marketing Analytics',
    'Credit Risk', 'Market Risk', 'Operations',
    'Wealth Management', 'Card Services', 'Mortgage',
]

TEAMS = [
    'platform-core', 'etl-pipeline', 'ml-ops', 'analytics-eng',
    'feature-store', 'data-quality', 'streaming-ingest',
    'reporting', 'bi-team', 'quant-research', 'fraud-ml',
    'aml-detection', 'credit-scoring', 'nrt-analytics',
    'data-governance', 'lakehouse-admin', 'cost-optimization',
]

SKU_MAP = {
    'STANDARD_ALL_PURPOSE_COMPUTE': 0.55, 'PREMIUM_ALL_PURPOSE_COMPUTE': 0.70,
    'JOBS_COMPUTE': 0.15, 'JOBS_LIGHT_COMPUTE': 0.10,
    'SERVERLESS_SQL': 0.70, 'PRO_SQL': 0.55,
    'SERVERLESS_REAL_TIME_INFERENCE': 0.07, 'GPU_ALL_PURPOSE_COMPUTE': 1.50,
    'DLT_CORE_COMPUTE': 0.20, 'DLT_PRO_COMPUTE': 0.25, 'DLT_ADVANCED_COMPUTE': 0.36,
}

SKU_WEIGHTS = {
    'JOBS_COMPUTE': 0.35, 'SERVERLESS_SQL': 0.20, 'PRO_SQL': 0.10,
    'STANDARD_ALL_PURPOSE_COMPUTE': 0.08, 'PREMIUM_ALL_PURPOSE_COMPUTE': 0.05,
    'DLT_PRO_COMPUTE': 0.07, 'DLT_ADVANCED_COMPUTE': 0.04, 'DLT_CORE_COMPUTE': 0.03,
    'GPU_ALL_PURPOSE_COMPUTE': 0.04, 'SERVERLESS_REAL_TIME_INFERENCE': 0.02,
    'JOBS_LIGHT_COMPUTE': 0.02,
}

NODE_TYPES = [
    'i3.xlarge', 'i3.2xlarge', 'i3.4xlarge', 'i3.8xlarge',
    'r5.xlarge', 'r5.2xlarge', 'r5.4xlarge', 'r5.8xlarge',
    'm5.xlarge', 'm5.2xlarge', 'm5.4xlarge',
    'p3.2xlarge', 'p3.8xlarge', 'g4dn.xlarge', 'g4dn.4xlarge',
]

DBR_VERSIONS = [
    '12.2.x-scala2.12', '13.0.x-scala2.12', '13.3.x-scala2.12',
    '14.0.x-scala2.12', '14.3.x-scala2.12', '15.0.x-scala2.12',
    '15.2.x-scala2.12', '15.4.x-scala2.12',
]

FIRST_NAMES = [
    'James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda','David','Elizabeth',
    'William','Barbara','Richard','Susan','Joseph','Jessica','Thomas','Sarah','Christopher','Karen',
    'Charles','Lisa','Daniel','Nancy','Matthew','Betty','Anthony','Margaret','Mark','Sandra',
    'Donald','Ashley','Steven','Dorothy','Paul','Kimberly','Andrew','Emily','Joshua','Donna',
    'Kenneth','Michelle','Kevin','Carol','Brian','Amanda','George','Melissa','Timothy','Deborah',
    'Ronald','Stephanie','Edward','Rebecca','Jason','Sharon','Jeffrey','Laura','Ryan','Cynthia',
    'Jacob','Kathleen','Gary','Amy','Nicholas','Angela','Eric','Shirley','Jonathan','Anna',
    'Stephen','Brenda','Larry','Pamela','Justin','Emma','Scott','Nicole','Brandon','Helen',
    'Benjamin','Samantha','Samuel','Katherine','Raymond','Christine','Gregory','Debra','Frank','Rachel',
    'Alexander','Carolyn','Patrick','Janet','Jack','Catherine','Dennis','Maria','Jerry','Heather',
    'Tyler','Diane','Aaron','Ruth','Jose','Julie','Nathan','Olivia','Henry','Joyce',
    'Peter','Virginia','Douglas','Victoria','Zachary','Kelly','Kyle','Lauren','Noah','Christina',
    'Ethan','Joan','Adrian','Evelyn','Aiden','Judith','Dylan','Megan',
    'Priya','Raj','Anita','Vikram','Deepa','Sanjay','Neha','Amit',
    'Wei','Ming','Yuki','Hiro','Jin','Soo','Chen','Li',
]

LAST_NAMES = [
    'Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez','Martinez',
    'Hernandez','Lopez','Gonzalez','Wilson','Anderson','Thomas','Taylor','Moore','Jackson','Martin',
    'Lee','Perez','Thompson','White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson',
    'Walker','Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores','Green',
    'Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell','Carter','Roberts',
    'Patel','Shah','Kumar','Singh','Gupta','Sharma','Chen','Wang','Li','Zhang','Liu','Yang','Wu',
    'Kim','Park','Cho','Jung','Tanaka','Suzuki','Watanabe',
    "O'Brien",'Murphy','Sullivan','Cohen','Goldberg','Katz',
]

def sql_str(s):
    return s.replace("'", "''")

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# Generate users
def generate_users(count=1100):
    users, used = [], set()
    random.seed(42)
    for _ in range(count * 2):
        if len(users) >= count: break
        email = f"{random.choice(FIRST_NAMES).lower()}.{random.choice(LAST_NAMES).lower()}@example.com"
        if email not in used:
            used.add(email); users.append(email)
    return users

USERS = generate_users(1100)
NOW = datetime(2026, 4, 26)
THREE_YEARS_AGO = NOW - timedelta(days=1095)

random.seed(42)
USER_DEPT = {u: random.choice(DEPARTMENTS) for u in USERS}
USER_TEAM = {u: random.choice(TEAMS) for u in USERS}

print(f'Generated {len(USERS)} users')
print(f'Date range: {THREE_YEARS_AGO.strftime("%Y-%m-%d")} → {NOW.strftime("%Y-%m-%d")}')

In [ ]:
# ── Generate Cluster, Warehouse, Job definitions ─────────────────────

def generate_clusters(count=200):
    random.seed(100)
    clusters = []
    for i in range(count):
        ws = random.choice(WORKSPACES)
        owner = random.choice(USERS)
        cname = '-'.join([random.choice(['etl','analytics','ml','streaming','adhoc','prod','staging','dev']),
                          random.choice(['pipeline','cluster','compute','workload','processing']),
                          str(random.randint(1,50))])
        clusters.append(dict(
            ws=ws, id=f'cls-{i+1:04d}', name=cname, owner=owner,
            driver=random.choice(NODE_TYPES[:11]), worker=random.choice(NODE_TYPES[:11]),
            workers=random.choice([2,4,8,16,32]),
            min_w=max(1, random.choice([2,4,8,16,32])//4),
            max_w=random.choice([2,4,8,16,32])*2,
            auto_term=random.choice([60,120,240,0]),
            dbr=random.choice(DBR_VERSIONS), team=USER_TEAM[owner], dept=USER_DEPT[owner],
            days_ago=random.randint(30,1000), deleted=random.random()<0.15,
            security=random.choice(['SINGLE_USER','USER_ISOLATION','NO_ISOLATION'])))
    return clusters

def generate_warehouses(count=30):
    random.seed(200)
    whs = []
    sizes = ['2X-Small','X-Small','Small','Medium','Large','X-Large','2X-Large']
    for i in range(count):
        ws = random.choice(WORKSPACES)
        whs.append(dict(
            ws=ws, id=f'wh-{i+1:04d}',
            name=f"{random.choice(['reporting','bi','adhoc','etl','prod','staging'])}-warehouse-{i+1}",
            type=random.choice(['PRO','CLASSIC','SERVERLESS']),
            size=random.choice(sizes), min_c=random.choice([1,1,1,2]),
            max_c=random.choice([1,2,4,8,16]), auto_stop=random.choice([5,10,15,30]),
            days_ago=random.randint(30,900)))
    return whs

def generate_jobs(count=500):
    random.seed(300)
    jobs = []
    job_types = ['ETL-Daily','ETL-Hourly','ML-Training','ML-Scoring','Report-Generation',
                 'Data-Quality','Feature-Engineering','Streaming-Ingest','CDC-Pipeline',
                 'Archive-Job','Compliance-Check','AML-Scan','Fraud-Score','Risk-Calc',
                 'PnL-Report','Regulatory-Filing','Customer-360','Segmentation',
                 'Campaign-Analytics','Real-Time-Alerts']
    schedules = ['0 0 * * * ?','0 0 8 * * ?','0 0 6 * * ?','0 30 7 * * ?',
                 '0 0 0 * * ?','0 0 */4 * * ?','0 0 9 ? * MON']
    for i in range(count):
        ws = random.choice(WORKSPACES)
        creator = random.choice(USERS)
        jtype = random.choice(job_types)
        jobs.append(dict(
            ws=ws, id=f'job-{i+1:05d}',
            name=f"{jtype}-{USER_DEPT[creator].lower().replace(' ','-')}-{random.randint(1,99)}",
            creator=creator, schedule=random.choice(schedules),
            days_ago=random.randint(10,1000), deleted=random.random()<0.08,
            team=USER_TEAM[creator], env=random.choice(['prod','staging','dev'])))
    return jobs

CLUSTERS = generate_clusters()
WAREHOUSES = generate_warehouses()
JOBS = generate_jobs()

print(f'Clusters: {len(CLUSTERS)}, Warehouses: {len(WAREHOUSES)}, Jobs: {len(JOBS)}')

In [ ]:
# ── Step 1: Create all schemas ───────────────────────────────────────
print('Creating schemas...')
for schema in ['billing','access','compute','lakeflow','query','ai_gateway',
               'serving','mlflow','storage','information_schema',
               'networking','lakeview','dashboards','marketplace']:
    run_sql(f'schema {schema}', f'CREATE SCHEMA IF NOT EXISTS workspace.mock_system_{schema}')
print('Done!')

In [ ]:
# ── Step 2: billing.usage (3 years, ~$5M/year) ──────────────────────
print('Generating billing usage data (3 years)...')

run_sql('drop billing.usage', 'DROP TABLE IF EXISTS workspace.mock_system_billing.usage')
run_sql('create billing.usage', '''
CREATE TABLE workspace.mock_system_billing.usage (
  record_id STRING, account_id STRING, workspace_id STRING, sku_name STRING, cloud STRING,
  usage_start_time TIMESTAMP, usage_end_time TIMESTAMP, usage_date DATE,
  custom_tags MAP<STRING, STRING>, usage_unit STRING, usage_quantity DOUBLE,
  usage_type STRING, billing_origin_product STRING, record_type STRING, ingestion_date DATE,
  identity_metadata STRUCT<run_as: STRING, created_by: STRING>,
  usage_metadata STRUCT<cluster_id: STRING, warehouse_id: STRING, job_id: STRING,
    job_run_id: STRING, dlt_pipeline_id: STRING, notebook_id: STRING,
    endpoint_name: STRING, endpoint_id: STRING, run_id: STRING>
)
''')

random.seed(42)
rows = []
rc = 0
yearly_daily_target = {0: 9600, 1: 12300, 2: 15000}
current = THREE_YEARS_AGO

while current < NOW:
    yi = min(2, (current - THREE_YEARS_AGO).days // 365)
    dt = yearly_daily_target[yi]
    dow = current.weekday()
    if dow >= 5: dt = int(dt * 0.4)
    if current.day >= 28: dt = int(dt * 1.3)
    if current.month in (3,6,9,12) and current.day >= 25: dt = int(dt * 1.5)

    for sku, weight in SKU_WEIGHTS.items():
        sb = int(dt * weight)
        if sb < 1: continue
        price = SKU_MAP[sku]
        total_dbu = sb / price
        nr = random.randint(5, 25)
        for _ in range(nr):
            rc += 1
            user = random.choice(USERS)
            ws = random.choice(WORKSPACES)
            team = USER_TEAM[user]; dept = USER_DEPT[user]
            dbu = round(total_dbu / nr * random.uniform(0.5, 1.5), 2)
            ho = random.randint(0, 23); dm = random.randint(10, 180)
            us = current.replace(hour=ho, minute=0, second=0)
            ue = us + timedelta(minutes=dm)
            cid=wid=jid=jrid=dltid=nbid=epn=epi=rid=''
            if 'ALL_PURPOSE' in sku or 'GPU' in sku:
                cid = random.choice(CLUSTERS)['id']; nbid = f'nb-{random.randint(1,5000)}'
            elif 'SQL' in sku: wid = random.choice(WAREHOUSES)['id']
            elif 'JOBS' in sku: j=random.choice(JOBS); jid=j['id']; jrid=f'run-{rc}'
            elif 'DLT' in sku: dltid = f'dlt-{random.randint(1,100):03d}'
            elif 'INFERENCE' in sku:
                epn = random.choice(['fraud-scorer','credit-model','recommender','nlp-classifier'])
                epi = f'ep-{random.randint(1,20):03d}'
            bp = 'INTERACTIVE' if 'ALL_PURPOSE' in sku else ('SQL' if 'SQL' in sku else ('JOBS' if 'JOBS' in sku else ('DLT' if 'DLT' in sku else 'SERVING')))
            rows.append(
                f"('r-{rc:08d}','{ACCOUNT_ID}','{ws[0]}','{sku}','AWS',"
                f"'{us.strftime('%Y-%m-%d %H:%M:%S')}','{ue.strftime('%Y-%m-%d %H:%M:%S')}',"
                f"'{current.strftime('%Y-%m-%d')}',map('team','{sql_str(team)}','department','{sql_str(dept)}'),'DBU',{dbu},'{bp}',"
                f"'{bp}','ORIGINAL','{current.strftime('%Y-%m-%d')}',"
                f"named_struct('run_as','{sql_str(user)}','created_by','{sql_str(user)}'),"
                f"named_struct('cluster_id','{cid}','warehouse_id','{wid}','job_id','{jid}','job_run_id','{jrid}',"
                f"'dlt_pipeline_id','{dltid}','notebook_id','{nbid}','endpoint_name','{epn}','endpoint_id','{epi}','run_id','{rid}'))")
    current += timedelta(days=1)

print(f'Generated {len(rows)} billing rows. Inserting in chunks...')
for i, chunk in enumerate(chunk_list(rows, 500)):
    vals = ',\n'.join(chunk)
    run_sql(f'billing chunk {i+1}/{(len(rows)//500)+1}',
            f'INSERT INTO workspace.mock_system_billing.usage VALUES\n{vals}')
print(f'✅ billing.usage loaded: {len(rows)} rows')

In [ ]:
# ── Step 3: billing.list_prices ──────────────────────────────────────
run_sql('drop list_prices', 'DROP TABLE IF EXISTS workspace.mock_system_billing.list_prices')
run_sql('create list_prices', '''
CREATE TABLE workspace.mock_system_billing.list_prices (
  price_start_time TIMESTAMP, price_end_time TIMESTAMP, account_id STRING,
  sku_name STRING, cloud STRING, currency_code STRING, usage_unit STRING,
  pricing STRUCT<default: DOUBLE, promotional: DOUBLE, effective_list: DOUBLE>
)
''')
price_rows = []
for sku, price in SKU_MAP.items():
    price_rows.append(
        f"('{THREE_YEARS_AGO.strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP),"
        f"'{ACCOUNT_ID}','{sku}','AWS','USD','DBU',"
        f"named_struct('default',{price},'promotional',cast(null as DOUBLE),'effective_list',{price}))")
run_sql('insert list_prices', f"INSERT INTO workspace.mock_system_billing.list_prices VALUES\n" + ',\n'.join(price_rows))
print('✅ billing.list_prices loaded')

In [ ]:
# ── Step 4: access.workspaces_latest ─────────────────────────────────
run_sql('drop workspaces', 'DROP TABLE IF EXISTS workspace.mock_system_access.workspaces_latest')
run_sql('create workspaces', '''
CREATE TABLE workspace.mock_system_access.workspaces_latest (
  account_id STRING, workspace_id STRING, workspace_name STRING, workspace_url STRING,
  workspace_status STRING, cloud STRING, region STRING, pricing_tier STRING, creation_time TIMESTAMP
)
''')
ws_rows = [f"('{ACCOUNT_ID}','{wid}','{wn}','https://{wn}.cloud.databricks.com','RUNNING','AWS','{wr}','ENTERPRISE','{THREE_YEARS_AGO.strftime('%Y-%m-%d %H:%M:%S')}')" for wid,wn,wr in WORKSPACES]
run_sql('insert workspaces', f"INSERT INTO workspace.mock_system_access.workspaces_latest VALUES\n" + ',\n'.join(ws_rows))
print('✅ access.workspaces_latest loaded')

In [ ]:
# ── Step 5: access.audit (3 years) ───────────────────────────────────
print('Generating audit log data (3 years)...')

run_sql('drop audit', 'DROP TABLE IF EXISTS workspace.mock_system_access.audit')
run_sql('create audit', '''
CREATE TABLE workspace.mock_system_access.audit (
  account_id STRING, workspace_id STRING, version STRING, event_time TIMESTAMP,
  event_date DATE, source_ip_address STRING, user_agent STRING, session_id STRING,
  user_identity STRUCT<email: STRING, subjectName: STRING>,
  service_name STRING, action_name STRING, request_id STRING,
  request_params MAP<STRING, STRING>,
  response STRUCT<statusCode: INT, errorMessage: STRING, result: STRING>,
  audit_level STRING, event_id STRING,
  identity_metadata STRUCT<run_by: STRING, run_as: STRING>
)
''')

random.seed(55)
actions = [
    ('accounts','login'),('accounts','logout'),('clusters','create'),('clusters','start'),
    ('clusters','terminate'),('jobs','create'),('jobs','runNow'),('sql','commandSubmit'),
    ('sql','commandFinish'),('notebook','runCommand'),('databrickssql','getWarehouse'),
    ('secrets','getSecret'),('unityCatalog','getTable'),('unityCatalog','createTable'),
    ('mlflow','createRun'),('workspace','fileCreate'),('iamRole','changePermissions'),
]
audit_rows = []
ctr = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(30, 80)):
        ctr += 1
        user = random.choice(USERS); ws = random.choice(WORKSPACES)
        svc, act = random.choice(actions)
        et = cur.replace(hour=random.randint(6,22), minute=random.randint(0,59), second=random.randint(0,59))
        st = 200 if random.random() < 0.95 else random.choice([401,403,500])
        em = '' if st == 200 else 'Access denied'
        res = 'success' if st == 200 else 'failure'
        ip = f'{random.randint(10,172)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}'
        err_sql = 'cast(null as STRING)' if not em else repr(em)
        audit_rows.append(
            f"('{ACCOUNT_ID}','{ws[0]}','2.0','{et.strftime('%Y-%m-%d %H:%M:%S')}','{cur.strftime('%Y-%m-%d')}',"
            f"'{ip}','Databricks/API','sess-{ctr:08d}',"
            f"named_struct('email','{sql_str(user)}','subjectName','{sql_str(user)}'),"
            f"'{svc}','{act}','req-{ctr:08d}',map('user','{sql_str(user)}'),"
            f"named_struct('statusCode',{st},'errorMessage',{err_sql},'result','{res}'),"
            f"'WORKSPACE_LEVEL','evt-{ctr:08d}',"
            f"named_struct('run_by','{sql_str(user)}','run_as','{sql_str(user)}'))")
    cur += timedelta(days=1)

print(f'Generated {len(audit_rows)} audit rows. Inserting...')
for i, chunk in enumerate(chunk_list(audit_rows, 500)):
    run_sql(f'audit chunk {i+1}/{(len(audit_rows)//500)+1}', f'INSERT INTO workspace.mock_system_access.audit VALUES\n' + ',\n'.join(chunk))
print(f'✅ access.audit loaded: {len(audit_rows)} rows')

In [ ]:
# ── Step 6: compute.clusters + warehouses ────────────────────────────
run_sql('drop clusters', 'DROP TABLE IF EXISTS workspace.mock_system_compute.clusters')
run_sql('create clusters', '''
CREATE TABLE workspace.mock_system_compute.clusters (
  account_id STRING, workspace_id STRING, cluster_id STRING, cluster_name STRING,
  owned_by STRING, create_time TIMESTAMP, delete_time TIMESTAMP,
  driver_node_type STRING, worker_node_type STRING, worker_count BIGINT,
  min_autoscale_workers BIGINT, max_autoscale_workers BIGINT,
  auto_termination_minutes BIGINT, enable_elastic_disk BOOLEAN,
  tags MAP<STRING, STRING>, cluster_source STRING, dbr_version STRING,
  change_time TIMESTAMP, change_date DATE, data_security_mode STRING
)
''')

cl_rows = []
for c in CLUSTERS:
    ws = c['ws']
    cd = (NOW - timedelta(days=c['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    dt_str = f"'{(NOW - timedelta(days=random.randint(1, c['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}'" if c['deleted'] else 'cast(null as TIMESTAMP)'
    chd = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d')
    cl_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{c['id']}','{c['name']}','{sql_str(c['owner'])}',"
        f"'{cd}',{dt_str},'{c['driver']}','{c['worker']}',{c['workers']},{c['min_w']},{c['max_w']},"
        f"{c['auto_term']},true,map('team','{c['team']}','department','{c['dept']}'),'API','{c['dbr']}',"
        f"'{chd} 10:00:00','{chd}','{c['security']}')")

run_sql('insert clusters', f'INSERT INTO workspace.mock_system_compute.clusters VALUES\n' + ',\n'.join(cl_rows))

# Warehouses
run_sql('drop warehouses', 'DROP TABLE IF EXISTS workspace.mock_system_compute.warehouses')
run_sql('create warehouses', '''
CREATE TABLE workspace.mock_system_compute.warehouses (
  account_id STRING, workspace_id STRING, warehouse_id STRING, warehouse_name STRING,
  warehouse_type STRING, warehouse_size STRING, min_clusters INT, max_clusters INT,
  auto_stop_minutes INT, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
wh_rows = []
for wh in WAREHOUSES:
    ws = wh['ws']
    ch = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')
    wh_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{wh['id']}','{wh['name']}','{wh['type']}','{wh['size']}',"
        f"{wh['min_c']},{wh['max_c']},{wh['auto_stop']},'{ch}',cast(null as TIMESTAMP))")
run_sql('insert warehouses', f'INSERT INTO workspace.mock_system_compute.warehouses VALUES\n' + ',\n'.join(wh_rows))

# Empty event tables
for tbl, ddl in [
    ('cluster_events', 'account_id STRING,workspace_id STRING,cluster_id STRING,timestamp TIMESTAMP,type STRING,details MAP<STRING,STRING>'),
    ('warehouse_events', 'account_id STRING,workspace_id STRING,warehouse_id STRING,event_type STRING,cluster_count INT,event_time TIMESTAMP'),
]:
    run_sql(f'drop {tbl}', f'DROP TABLE IF EXISTS workspace.mock_system_compute.{tbl}')
    run_sql(f'create {tbl}', f'CREATE TABLE workspace.mock_system_compute.{tbl} ({ddl})')

print(f'✅ compute tables loaded: {len(CLUSTERS)} clusters, {len(WAREHOUSES)} warehouses')

In [ ]:
# ── Step 7: compute.node_timeline (90 days) ─────────────────────────
print('Generating node timeline (90 days)...')
run_sql('drop node_timeline', 'DROP TABLE IF EXISTS workspace.mock_system_compute.node_timeline')
run_sql('create node_timeline', '''
CREATE TABLE workspace.mock_system_compute.node_timeline (
  account_id STRING, workspace_id STRING, cluster_id STRING, node_id STRING,
  instance_id STRING, start_time TIMESTAMP, end_time TIMESTAMP,
  driver BOOLEAN, is_driver BOOLEAN,
  cpu_user_percent DOUBLE, cpu_system_percent DOUBLE, cpu_wait_percent DOUBLE,
  mem_used_percent DOUBLE, mem_swap_percent DOUBLE,
  network_sent_bytes BIGINT, network_received_bytes BIGINT,
  node_type STRING, private_ip STRING, uptime_seconds DOUBLE,
  num_task_slots INT, avg_num_running_tasks DOUBLE, avg_num_queued_tasks DOUBLE
)
''')

random.seed(111)
node_rows = []
for day_off in range(90):
    cd = NOW - timedelta(days=day_off)
    for cl in random.sample(CLUSTERS, min(40, len(CLUSTERS))):
        ws = cl['ws']
        for ni in range(random.randint(2, cl['workers']+1)):
            isd = (ni == 0)
            cpu = random.uniform(5,95); cpus = random.uniform(1,15)
            mem = random.uniform(20,95)
            st = cd.replace(hour=random.randint(6,22), minute=0, second=0)
            en = st + timedelta(hours=random.randint(1,8))
            node_rows.append(
                f"('{ACCOUNT_ID}','{ws[0]}','{cl['id']}','node-{day_off:03d}-{cl['id']}-{ni}',"
                f"'i-{random.randint(10000,99999):05d}{random.randint(10000,99999):05d}',"
                f"'{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}',"
                f"{str(isd).lower()},{str(isd).lower()},{cpu:.1f},{cpus:.1f},{random.uniform(0,10):.1f},"
                f"{mem:.1f},{random.uniform(0,2):.1f},{random.randint(100000,10000000000)},"
                f"{random.randint(100000,10000000000)},'{cl['worker']}',"
                f"'10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',"
                f"{random.uniform(3600,28800):.0f},{random.randint(4,32)},"
                f"{random.uniform(0.5,28):.1f},{random.uniform(0,5):.1f})")

print(f'Generated {len(node_rows)} node rows. Inserting...')
for i, chunk in enumerate(chunk_list(node_rows, 500)):
    run_sql(f'nodes chunk {i+1}', f'INSERT INTO workspace.mock_system_compute.node_timeline VALUES\n' + ',\n'.join(chunk))
print(f'✅ compute.node_timeline loaded: {len(node_rows)} rows')

In [ ]:
# ── Step 8: lakeflow.jobs + job_run_timeline (3 years) ───────────────
print('Loading lakeflow tables...')

run_sql('drop jobs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.jobs')
run_sql('create jobs', '''
CREATE TABLE workspace.mock_system_lakeflow.jobs (
  account_id STRING, workspace_id STRING, job_id STRING, name STRING,
  creator_user_name STRING, run_as_user_name STRING, tags MAP<STRING, STRING>,
  schedule STRUCT<quartz_cron_expression: STRING, pause_status: STRING>,
  created_time TIMESTAMP, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
j_rows = []
for j in JOBS:
    ws = j['ws']
    cr = (NOW - timedelta(days=j['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    ch = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')
    dt = f"'{(NOW - timedelta(days=random.randint(1,j['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}'" if j['deleted'] else 'cast(null as TIMESTAMP)'
    pa = 'UNPAUSED' if random.random()<0.85 else 'PAUSED'
    j_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','{sql_str(j['name'])}',"
        f"'{sql_str(j['creator'])}','{sql_str(j['creator'])}',"
        f"map('Env','{j['env']}','team','{j['team']}'),"
        f"named_struct('quartz_cron_expression','{j['schedule']}','pause_status','{pa}'),"
        f"'{cr}','{ch}',{dt})")
for i, chunk in enumerate(chunk_list(j_rows, 200)):
    run_sql(f'jobs chunk {i+1}', f'INSERT INTO workspace.mock_system_lakeflow.jobs VALUES\n' + ',\n'.join(chunk))

# Job run timeline
print('Generating job run timeline (3 years)...')
run_sql('drop job_runs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_run_timeline')
run_sql('create job_runs', '''
CREATE TABLE workspace.mock_system_lakeflow.job_run_timeline (
  account_id STRING, workspace_id STRING, job_id STRING, run_id STRING,
  period_start_time TIMESTAMP, period_end_time TIMESTAMP,
  trigger_type STRING, run_type STRING, result_state STRING, termination_code STRING
)
''')

random.seed(88)
jr_rows = []
jrc = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(80, 300)):
        jrc += 1
        j = random.choice(JOBS); ws = j['ws']
        st = cur.replace(hour=random.randint(0,23), minute=random.randint(0,59))
        dur = random.randint(2, 240); en = st + timedelta(minutes=dur)
        rs = random.choices(['SUCCEEDED','FAILED','CANCELLED','TIMED_OUT'], weights=[.88,.07,.03,.02])[0]
        tc = 'SUCCESS' if rs=='SUCCEEDED' else ('RUN_EXECUTION_ERROR' if rs=='FAILED' else ('USER_CANCELLED' if rs=='CANCELLED' else 'MAX_RUN_DURATION_EXCEEDED'))
        tr = random.choice(['SCHEDULED','MANUAL','RETRY','FILE_ARRIVAL'])
        jr_rows.append(
            f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','run-{jrc:08d}',"
            f"'{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}',"
            f"'{tr}','JOB_RUN','{rs}','{tc}')")
    cur += timedelta(days=1)

print(f'Generated {len(jr_rows)} job run rows. Inserting...')
for i, chunk in enumerate(chunk_list(jr_rows, 500)):
    run_sql(f'job_runs chunk {i+1}/{(len(jr_rows)//500)+1}', f'INSERT INTO workspace.mock_system_lakeflow.job_run_timeline VALUES\n' + ',\n'.join(chunk))

# Job tasks + task run timeline + pipelines (lightweight)
run_sql('drop job_tasks', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_tasks')
run_sql('create job_tasks', 'CREATE TABLE workspace.mock_system_lakeflow.job_tasks (account_id STRING,workspace_id STRING,job_id STRING,task_key STRING,change_time TIMESTAMP,delete_time TIMESTAMP)')
tk_rows = []
for j in JOBS[:200]:
    ws = j['ws']
    for t in range(random.randint(1,8)):
        tk_rows.append(f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','task-{t+1}','{(NOW-timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP))")
for i, chunk in enumerate(chunk_list(tk_rows, 500)):
    run_sql(f'tasks chunk {i+1}', f'INSERT INTO workspace.mock_system_lakeflow.job_tasks VALUES\n' + ',\n'.join(chunk))

run_sql('drop task_runs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_task_run_timeline')
run_sql('create task_runs', 'CREATE TABLE workspace.mock_system_lakeflow.job_task_run_timeline (account_id STRING,workspace_id STRING,job_id STRING,run_id STRING,task_key STRING,period_start_time TIMESTAMP,period_end_time TIMESTAMP,result_state STRING,termination_code STRING)')

# Pipelines
run_sql('drop pipelines', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.pipelines')
run_sql('create pipelines', '''
CREATE TABLE workspace.mock_system_lakeflow.pipelines (
  account_id STRING, workspace_id STRING, pipeline_id STRING, name STRING,
  creator_user_name STRING, run_as_user_name STRING, channel STRING,
  edition STRING, created_time TIMESTAMP, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
pnames = ['CDC-Customer-Accounts','Fraud-Alerts-Stream','Transaction-ETL','Risk-Score-Pipeline',
          'AML-Detection','Market-Data-Ingest','Regulatory-Reporting','Customer-360-Build',
          'Credit-Score-Update','Payment-Processing','Loan-Origination','Card-Transaction-Stream',
          'KYC-Verification','Portfolio-Analytics','Trade-Settlement','Compliance-Audit-Trail',
          'Digital-Banking-Events','ATM-Transaction-Stream','Branch-Performance','Wealth-Portfolio-Sync']
p_rows = []
for i, pn in enumerate(pnames):
    ws = WORKSPACES[i % len(WORKSPACES)]; cr = random.choice(USERS)
    ed = random.choice(['CORE','PRO','ADVANCED'])
    p_rows.append(f"('{ACCOUNT_ID}','{ws[0]}','dlt-{i+1:03d}','{pn}','{sql_str(cr)}','{sql_str(cr)}','CURRENT','{ed}','{(NOW-timedelta(days=random.randint(100,900))).strftime('%Y-%m-%d %H:%M:%S')}','{(NOW-timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP))")
run_sql('insert pipelines', f'INSERT INTO workspace.mock_system_lakeflow.pipelines VALUES\n' + ',\n'.join(p_rows))

run_sql('drop pipe_updates', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.pipeline_update_timeline')
run_sql('create pipe_updates', 'CREATE TABLE workspace.mock_system_lakeflow.pipeline_update_timeline (account_id STRING,workspace_id STRING,pipeline_id STRING,update_id STRING,result_state STRING,period_start_time TIMESTAMP,period_end_time TIMESTAMP)')

print(f'✅ lakeflow loaded: {len(JOBS)} jobs, {len(jr_rows)} runs, {len(tk_rows)} tasks, {len(pnames)} pipelines')

In [ ]:
# ── Step 9: query.history ────────────────────────────────────────────
print('Generating query history...')
run_sql('drop query_history', 'DROP TABLE IF EXISTS workspace.mock_system_query.history')
run_sql('create query_history', '''
CREATE TABLE workspace.mock_system_query.history (
  statement_id STRING, executed_by STRING, executed_as STRING,
  compute STRUCT<warehouse_id: STRING>, statement_type STRING, statement_text STRING,
  start_time TIMESTAMP, total_duration_ms BIGINT, read_bytes BIGINT,
  read_rows BIGINT, error_message STRING
)
''')

random.seed(77)
queries_tmpl = [
    ('SELECT','SELECT * FROM transactions WHERE date >= current_date - 30'),
    ('SELECT','SELECT customer_id, SUM(amount) FROM payments GROUP BY 1'),
    ('SELECT','SELECT risk_score, COUNT(*) FROM credit_assessments GROUP BY 1'),
    ('INSERT','INSERT INTO daily_aggregates SELECT date, SUM(volume) FROM trades GROUP BY 1'),
    ('MERGE','MERGE INTO customer_360 USING staging_customers ON id = s_id'),
    ('SELECT','SELECT * FROM fraud_alerts WHERE score > 0.9 AND created_date = current_date'),
    ('SELECT','SELECT acct_type, AVG(balance) FROM accounts GROUP BY 1'),
    ('SELECT','SELECT COUNT(DISTINCT customer_id) FROM digital_banking_events'),
    ('SELECT','SELECT department, SUM(cost) FROM cost_allocation GROUP BY 1'),
]
q_rows = []
qc = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(100, 400)):
        qc += 1
        user = random.choice(USERS); wh = random.choice(WAREHOUSES)
        st_type, st_text = random.choice(queries_tmpl)
        start = cur.replace(hour=random.randint(7,21), minute=random.randint(0,59))
        dur = random.randint(50, 300000)
        he = random.random() < 0.02
        q_rows.append(
            f"('stmt-{qc:08d}','{sql_str(user)}','{sql_str(user)}',"
            f"named_struct('warehouse_id','{wh['id']}'),'{st_type}','{sql_str(st_text)}',"
            f"'{start.strftime('%Y-%m-%d %H:%M:%S')}',{dur},{random.randint(1000,10737418240)},"
            f"{random.randint(100,50000000)},{'cast(null as STRING)' if not he else repr('TABLE_OR_VIEW_NOT_FOUND')})")
    cur += timedelta(days=7)  # weekly sample

print(f'Generated {len(q_rows)} query rows. Inserting...')
for i, chunk in enumerate(chunk_list(q_rows, 500)):
    run_sql(f'queries chunk {i+1}', f'INSERT INTO workspace.mock_system_query.history VALUES\n' + ',\n'.join(chunk))
print(f'✅ query.history loaded: {len(q_rows)} rows')

In [ ]:
# ── Step 10: ai_gateway.usage (1.5 years, growing) ──────────────────
print('Generating AI gateway usage...')
run_sql('drop ai_gateway', 'DROP TABLE IF EXISTS workspace.mock_system_ai_gateway.usage')
run_sql('create ai_gateway', '''
CREATE TABLE workspace.mock_system_ai_gateway.usage (
  route_name STRING, event_time TIMESTAMP, total_token_count BIGINT,
  input_token_count BIGINT, output_token_count BIGINT,
  execution_duration_ms DOUBLE, requester STRING
)
''')

random.seed(99)
routes = ['gpt-4-turbo','claude-3-sonnet','databricks-dbrx','databricks-meta-llama-3',
          'databricks-mixtral','text-embedding-ada-002','databricks-bge-large']
ai_rows = []
ai_start = NOW - timedelta(days=540)
cur = ai_start
while cur < NOW:
    n = random.randint(50, 300)
    gf = 1 + ((cur - ai_start).days / 540) * 3
    n = int(n * gf)
    for _ in range(min(n, 500)):
        user = random.choice(USERS[:400]); route = random.choice(routes)
        et = cur.replace(hour=random.randint(8,20), minute=random.randint(0,59))
        inp = random.randint(100,8000); out = random.randint(50,4000)
        ai_rows.append(
            f"('{route}','{et.strftime('%Y-%m-%d %H:%M:%S')}',{inp+out},{inp},{out},"
            f"{random.uniform(100,5000):.1f},'{sql_str(user)}')")
    cur += timedelta(days=1)

print(f'Generated {len(ai_rows)} AI rows. Inserting...')
for i, chunk in enumerate(chunk_list(ai_rows, 500)):
    run_sql(f'ai chunk {i+1}', f'INSERT INTO workspace.mock_system_ai_gateway.usage VALUES\n' + ',\n'.join(chunk))
print(f'✅ ai_gateway.usage loaded: {len(ai_rows)} rows')

In [ ]:
# ── Step 11: serving, mlflow, lineage, storage, info_schema ─────────
print('Loading remaining tables...')

# ── served_entities
run_sql('drop served_entities', 'DROP TABLE IF EXISTS workspace.mock_system_serving.served_entities')
run_sql('create served_entities', 'CREATE TABLE workspace.mock_system_serving.served_entities (endpoint_name STRING,served_entity_name STRING,entity_type STRING,workspace_id STRING,change_time TIMESTAMP)')
endpoints = [('fraud-scorer','fraud-detection-v3','CUSTOM_MODEL'),('credit-model','credit-risk-xgb','CUSTOM_MODEL'),
    ('recommender','product-recommender-v2','CUSTOM_MODEL'),('nlp-classifier','doc-classifier-bert','CUSTOM_MODEL'),
    ('llm-gateway','databricks-meta-llama-3','FOUNDATION_MODEL'),('embedding-service','databricks-bge-large','FOUNDATION_MODEL'),
    ('aml-detector','aml-detection-ensemble','CUSTOM_MODEL'),('kyc-verifier','kyc-document-ocr','CUSTOM_MODEL')]
ep_rows = [f"('{n}','{e}','{t}','{random.choice(WORKSPACES)[0]}','{(NOW-timedelta(days=random.randint(30,300))).strftime('%Y-%m-%d %H:%M:%S')}')" for n,e,t in endpoints]
run_sql('insert served_entities', f'INSERT INTO workspace.mock_system_serving.served_entities VALUES\n' + ',\n'.join(ep_rows))

# ── endpoint_usage (1 year)
run_sql('drop endpoint_usage', 'DROP TABLE IF EXISTS workspace.mock_system_serving.endpoint_usage')
run_sql('create endpoint_usage', 'CREATE TABLE workspace.mock_system_serving.endpoint_usage (served_entity_name STRING,request_time TIMESTAMP,total_token_count BIGINT,status_code INT)')
random.seed(133)
eu_rows = []
for d_off in range(365):
    d = NOW - timedelta(days=d_off)
    for _,entity,etype in endpoints:
        for _ in range(min(random.randint(50,500), 100)):
            t = d.replace(hour=random.randint(0,23), minute=random.randint(0,59))
            tok = random.randint(100,5000) if 'FOUNDATION' in etype else random.randint(0,10)
            st = 200 if random.random()<0.97 else random.choice([400,500,503])
            eu_rows.append(f"('{entity}','{t.strftime('%Y-%m-%d %H:%M:%S')}',{tok},{st})")
print(f'Endpoint usage: {len(eu_rows)} rows')
for i, chunk in enumerate(chunk_list(eu_rows, 500)):
    run_sql(f'ep_usage chunk {i+1}', f'INSERT INTO workspace.mock_system_serving.endpoint_usage VALUES\n' + ',\n'.join(chunk))

# ── MLflow experiments
run_sql('drop experiments', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.experiments_latest')
run_sql('create experiments', 'CREATE TABLE workspace.mock_system_mlflow.experiments_latest (experiment_id STRING,name STRING,lifecycle_stage STRING,creation_time BIGINT,last_update_time BIGINT)')
ml_exps = ['Fraud-Detection-V3','Credit-Risk-XGBoost','Customer-Churn-Prediction','Product-Recommender',
    'NLP-Document-Classifier','AML-Ensemble','Transaction-Anomaly','Loan-Default-Prediction',
    'Sentiment-Analysis','Market-Risk-VAR','Portfolio-Optimization','KYC-OCR-Model',
    'Card-Fraud-RealTime','Customer-LTV','Cross-Sell-Propensity','Branch-Demand-Forecast',
    'ATM-Cash-Optimization','Interest-Rate-Model','Mortgage-Prepayment','Collections-Priority']
exp_rows = []
for i, en in enumerate(ml_exps):
    ct = int((NOW-timedelta(days=random.randint(90,900))).timestamp()*1000)
    ut = int((NOW-timedelta(days=random.randint(1,60))).timestamp()*1000)
    stg = 'active' if random.random()<0.85 else 'deleted'
    exp_rows.append(f"('exp-{i+1:03d}','{en}','{stg}',{ct},{ut})")
run_sql('insert experiments', f'INSERT INTO workspace.mock_system_mlflow.experiments_latest VALUES\n' + ',\n'.join(exp_rows))

# ── MLflow runs
run_sql('drop ml_runs', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.runs_latest')
run_sql('create ml_runs', 'CREATE TABLE workspace.mock_system_mlflow.runs_latest (experiment_id STRING,run_id STRING,status STRING,start_time TIMESTAMP,end_time TIMESTAMP,user_id STRING)')
random.seed(144)
mr_rows = []
for i in range(len(ml_exps)):
    for r in range(random.randint(20,200)):
        st = NOW - timedelta(days=random.randint(1,365), hours=random.randint(0,23))
        en = st + timedelta(minutes=random.randint(5,480))
        status = random.choices(['FINISHED','FAILED','RUNNING','KILLED'],weights=[.75,.12,.08,.05])[0]
        mr_rows.append(f"('exp-{i+1:03d}','mlrun-{i+1:03d}-{r+1:04d}','{status}','{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}','{sql_str(random.choice(USERS[:300]))}')")
for i, chunk in enumerate(chunk_list(mr_rows, 500)):
    run_sql(f'ml_runs chunk {i+1}', f'INSERT INTO workspace.mock_system_mlflow.runs_latest VALUES\n' + ',\n'.join(chunk))

# ── Registered models
run_sql('drop models', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.registered_models_latest')
run_sql('create models', 'CREATE TABLE workspace.mock_system_mlflow.registered_models_latest (name STRING,creation_timestamp BIGINT,last_updated_timestamp BIGINT,user_id STRING)')
reg_models = ['fraud-detection-v3','credit-risk-xgb','customer-churn-model','product-recommender-v2',
    'doc-classifier-bert','aml-detection-ensemble','transaction-anomaly-v1','loan-default-pred',
    'sentiment-analyzer','market-risk-var','portfolio-optimizer','kyc-document-ocr']
rm_rows = [f"('{m}',{int((NOW-timedelta(days=random.randint(30,600))).timestamp()*1000)},{int((NOW-timedelta(days=random.randint(1,30))).timestamp()*1000)},'{sql_str(random.choice(USERS[:200]))}')" for m in reg_models]
run_sql('insert models', f'INSERT INTO workspace.mock_system_mlflow.registered_models_latest VALUES\n' + ',\n'.join(rm_rows))

print(f'✅ serving loaded: {len(endpoints)} endpoints, {len(eu_rows)} usage rows')
print(f'✅ mlflow loaded: {len(ml_exps)} experiments, {len(mr_rows)} runs, {len(reg_models)} models')

In [ ]:
# ── Step 12: lineage, storage, info_schema, marketplace ─────────────
print('Loading lineage, storage, and remaining tables...')

# Table lineage
run_sql('drop tbl_lineage', 'DROP TABLE IF EXISTS workspace.mock_system_access.table_lineage')
run_sql('create tbl_lineage', 'CREATE TABLE workspace.mock_system_access.table_lineage (source_table_catalog STRING,source_table_schema STRING,source_table_name STRING,target_table_catalog STRING,target_table_schema STRING,target_table_name STRING,event_time TIMESTAMP)')
lin_pairs = [('raw_transactions','bronze_transactions'),('bronze_transactions','silver_transactions'),
    ('silver_transactions','gold_transaction_summary'),('raw_customers','bronze_customers'),
    ('bronze_customers','silver_customer_360'),('silver_customer_360','gold_customer_analytics'),
    ('raw_market_data','bronze_market_data'),('bronze_market_data','silver_risk_metrics'),
    ('silver_risk_metrics','gold_risk_dashboard'),('raw_fraud_events','bronze_fraud_alerts'),
    ('bronze_fraud_alerts','silver_fraud_scores'),('silver_fraud_scores','gold_fraud_summary'),
    ('raw_loan_applications','bronze_loan_data'),('bronze_loan_data','silver_credit_assessment'),
    ('silver_credit_assessment','gold_lending_analytics')]
lin_rows = []
for src, tgt in lin_pairs:
    for d in range(30):
        dt = (NOW - timedelta(days=d)).strftime('%Y-%m-%d')
        lin_rows.append(f"('workspace','banking_data','{src}','workspace','banking_data','{tgt}','{dt} 08:00:00')")
run_sql('insert lineage', f'INSERT INTO workspace.mock_system_access.table_lineage VALUES\n' + ',\n'.join(lin_rows))

# Column lineage
run_sql('drop col_lineage', 'DROP TABLE IF EXISTS workspace.mock_system_access.column_lineage')
run_sql('create col_lineage', 'CREATE TABLE workspace.mock_system_access.column_lineage (source_table_catalog STRING,source_table_schema STRING,source_table_name STRING,source_column_name STRING,target_table_catalog STRING,target_table_schema STRING,target_table_name STRING,target_column_name STRING,event_time TIMESTAMP)')
col_pairs = [('raw_transactions','amount','silver_transactions','total_amount'),
    ('raw_transactions','customer_id','silver_transactions','customer_id'),
    ('raw_customers','email','silver_customer_360','contact_email'),
    ('raw_fraud_events','score','silver_fraud_scores','risk_score')]
cl_rows2 = [f"('workspace','banking_data','{s}','{sc}','workspace','banking_data','{t}','{tc}','{(NOW-timedelta(days=1)).strftime('%Y-%m-%d')} 08:00:00')" for s,sc,t,tc in col_pairs]
run_sql('insert col_lineage', f'INSERT INTO workspace.mock_system_access.column_lineage VALUES\n' + ',\n'.join(cl_rows2))

# Storage optimization
run_sql('drop storage_opt', 'DROP TABLE IF EXISTS workspace.mock_system_storage.predictive_optimization_operations_history')
run_sql('create storage_opt', 'CREATE TABLE workspace.mock_system_storage.predictive_optimization_operations_history (catalog_name STRING,schema_name STRING,table_name STRING,operation_type STRING,operation_status STRING,start_time TIMESTAMP,end_time TIMESTAMP,operation_metrics MAP<STRING,STRING>)')
st_tables = ['transactions','customers','accounts','payments','loans','credit_scores','market_data','trade_history','fraud_alerts']
st_rows = []
for day in range(90):
    d = NOW - timedelta(days=day)
    for tbl in random.sample(st_tables, random.randint(2,6)):
        op = random.choice(['OPTIMIZE','VACUUM','ZORDER'])
        sts = 'SUCCEEDED' if random.random()<0.92 else 'FAILED'
        s = d.replace(hour=2, minute=random.randint(0,59))
        e = s + timedelta(minutes=random.randint(3,45))
        st_rows.append(f"('workspace','banking_data','{tbl}','{op}','{sts}','{s.strftime('%Y-%m-%d %H:%M:%S')}','{e.strftime('%Y-%m-%d %H:%M:%S')}',map('files_removed','{random.randint(1,500)}','bytes_removed','{random.randint(10000,50000000000)}'))")
run_sql('insert storage', f'INSERT INTO workspace.mock_system_storage.predictive_optimization_operations_history VALUES\n' + ',\n'.join(st_rows))

# Information schema
run_sql('drop info_tables', 'DROP TABLE IF EXISTS workspace.mock_system_information_schema.tables')
run_sql('create info_tables', 'CREATE TABLE workspace.mock_system_information_schema.tables (table_catalog STRING,table_schema STRING,table_name STRING,table_type STRING,created TIMESTAMP)')
info_rows = [f"('workspace','banking_data','{t}','MANAGED','{(NOW-timedelta(days=900)).strftime('%Y-%m-%d %H:%M:%S')}')" for t in st_tables]
run_sql('insert info_tables', f'INSERT INTO workspace.mock_system_information_schema.tables VALUES\n' + ',\n'.join(info_rows))

# Marketplace
run_sql('drop mkt_listings', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listings')
run_sql('create mkt_listings', 'CREATE TABLE workspace.mock_system_marketplace.listings (listing_id STRING,listing_name STRING,provider STRING,category STRING,status STRING,created_time TIMESTAMP)')
listings = [('Market Risk Data Feed','Bloomberg','DATA'),('Fraud Intelligence','LexisNexis','DATA'),
    ('Credit Bureau Scores','Experian','DATA'),('Economic Indicators','Federal Reserve','DATA'),
    ('Geospatial Banking Data','SafeGraph','DATA')]
mkt_rows = [f"('mkt-{i+1:03d}','{n}','{p}','{c}','ACTIVE','{(NOW-timedelta(days=random.randint(90,700))).strftime('%Y-%m-%d %H:%M:%S')}')" for i,(n,p,c) in enumerate(listings)]
run_sql('insert mkt', f'INSERT INTO workspace.mock_system_marketplace.listings VALUES\n' + ',\n'.join(mkt_rows))

run_sql('drop mkt_access', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listing_access')
run_sql('create mkt_access', 'CREATE TABLE workspace.mock_system_marketplace.listing_access (listing_id STRING,consumer_id STRING,access_type STRING,access_time TIMESTAMP)')
run_sql('drop mkt_funnel', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listing_funnel_events')
run_sql('create mkt_funnel', 'CREATE TABLE workspace.mock_system_marketplace.listing_funnel_events (listing_id STRING,event_type STRING,event_time TIMESTAMP,consumer STRING)')

# Networking / Lakeview / Dashboards (empty structure)
for schema_tbl, ddl in [
    ('networking.private_endpoint_rules', 'account_id STRING,workspace_id STRING,rule_name STRING,resource_type STRING,status STRING,created_time TIMESTAMP'),
    ('networking.firewall_rules', 'account_id STRING,workspace_id STRING,rule_name STRING,cidr_block STRING,status STRING,created_time TIMESTAMP'),
    ('lakeview.dashboards', 'dashboard_id STRING,name STRING,creator STRING,created_time TIMESTAMP,updated_time TIMESTAMP'),
    ('lakeview.dashboard_usage', 'dashboard_id STRING,user_email STRING,view_time TIMESTAMP'),
    ('dashboards.dashboards', 'dashboard_id STRING,name STRING,creator STRING,created_time TIMESTAMP,updated_time TIMESTAMP'),
    ('dashboards.dashboard_usage', 'dashboard_id STRING,user_email STRING,view_time TIMESTAMP'),
]:
    full = f'workspace.mock_system_{schema_tbl}'
    run_sql(f'drop {schema_tbl}', f'DROP TABLE IF EXISTS {full}')
    run_sql(f'create {schema_tbl}', f'CREATE TABLE {full} ({ddl})')

print(f'✅ All remaining tables loaded!')
print(f'   Lineage: {len(lin_pairs)} table pairs × 30 days')
print(f'   Storage ops: {len(st_rows)} optimization records')
print(f'   Marketplace: {len(listings)} listings')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────
print('\n' + '='*70)
print('🎉 ENTERPRISE MOCK DATA LOAD COMPLETE')
print('='*70)
print(f'''
📊 Data Summary:
   Users:              {len(USERS):,}
   Departments:        {len(DEPARTMENTS)}
   Teams:              {len(TEAMS)}
   Workspaces:         {len(WORKSPACES)}
   Clusters:           {len(CLUSTERS):,}
   Warehouses:         {len(WAREHOUSES)}
   Jobs:               {len(JOBS):,}
   DLT Pipelines:      {len(pnames)}
   ML Experiments:     {len(ml_exps)}
   Registered Models:  {len(reg_models)}
   Serving Endpoints:  {len(endpoints)}
   
💰 Billing:
   Year 1 target:      ~$3.5M
   Year 2 target:      ~$4.5M
   Year 3 target:      ~$5.5M (forecast)
   Date range:         {THREE_YEARS_AGO.strftime("%Y-%m-%d")} → {NOW.strftime("%Y-%m-%d")}

📋 Tables loaded in workspace.mock_system_* schemas:
   billing.usage, billing.list_prices
   access.audit, access.workspaces_latest
   access.table_lineage, access.column_lineage
   compute.clusters, compute.warehouses, compute.node_timeline
   compute.cluster_events, compute.warehouse_events
   lakeflow.jobs, lakeflow.job_run_timeline, lakeflow.job_tasks
   lakeflow.job_task_run_timeline, lakeflow.pipelines, lakeflow.pipeline_update_timeline
   query.history
   ai_gateway.usage
   serving.served_entities, serving.endpoint_usage
   mlflow.experiments_latest, mlflow.runs_latest, mlflow.registered_models_latest
   storage.predictive_optimization_operations_history
   information_schema.tables
   marketplace.listings, marketplace.listing_access, marketplace.listing_funnel_events
   + networking, lakeview, dashboards tables
''')

## ☁️ Step 13: Cloud Platform Cost Mock Data (Azure / AWS / GCP)

Generates 90 days of realistic multi-cloud billing data modelled after a large enterprise
running a **hybrid multi-cloud strategy** (primary Azure, secondary AWS, GCP for AI/ML).

Writes to `workspace.platform.cloud_platform_costs` — the same table the Platform Cost dashboard tab reads from.

| Cloud | Accounts | Monthly spend |
|-------|----------|--------------|
| Azure | 3 subscriptions | ~$420k |
| AWS   | 3 accounts       | ~$210k |
| GCP   | 2 projects       | ~$95k  |

In [ ]:
import random, json as _json
from datetime import datetime, timedelta

AZURE_SUBSCRIPTIONS = [
    ('sub-prod-001',  'Production',    'eastus'),
    ('sub-dev-002',   'Development',   'westeurope'),
    ('sub-data-003',  'Data Platform', 'eastus2'),
]
AWS_ACCOUNTS = [
    ('111122223333', 'corp-prod-aws',  'us-east-1'),
    ('444455556666', 'corp-dev-aws',   'us-west-2'),
    ('777788889999', 'corp-data-aws',  'eu-west-1'),
]
GCP_PROJECTS = [
    ('corp-prod-gcp', 'Corp Production GCP',  'us-central1'),
    ('corp-ml-gcp',   'Corp ML Platform GCP', 'us-east1'),
]

# Tags pool — realistic enterprise resource tagging
TEAMS        = ['platform-core','etl-pipeline','ml-ops','analytics-eng','feature-store',
                'data-quality','streaming-ingest','reporting','bi-team','quant-research',
                'fraud-ml','aml-detection','credit-scoring','data-governance']
ENVS         = ['prod','prod','prod','staging','dev']      # prod-weighted
COST_CENTERS = [f'CC-{i:03d}' for i in range(1, 16)]
PROJECTS     = ['project-alpha','project-beta','lakehouse-migration','ml-platform',
                'data-mesh','compliance-reporting','real-time-fraud','customer-360']

def make_tags():
    return _json.dumps({
        'team':        random.choice(TEAMS),
        'environment': random.choice(ENVS),
        'cost_center': random.choice(COST_CENTERS),
        'project':     random.choice(PROJECTS),
    })

# Regional price multipliers (relative to base)
REGION_MUL = {
    'eastus': 1.0, 'eastus2': 1.0, 'westeurope': 1.12,
    'us-east-1': 1.0, 'us-west-2': 1.02, 'eu-west-1': 1.10,
    'us-central1': 1.0, 'us-east1': 1.0,
}

# Pricing model weights for compute vs non-compute
PM_COMPUTE  = [('ON_DEMAND',0.45),('RESERVED_1Y',0.30),('RESERVED_3Y',0.15),('SPOT',0.10)]
PM_DATA     = [('ON_DEMAND',0.70),('RESERVED_1Y',0.25),('RESERVED_3Y',0.05),('SPOT',0.00)]
PM_OTHER    = [('ON_DEMAND',1.00),('RESERVED_1Y',0.00),('RESERVED_3Y',0.00),('SPOT',0.00)]
RESERVED_DISC = {'ON_DEMAND':1.0,'RESERVED_1Y':0.62,'RESERVED_3Y':0.42,'SPOT':0.20}

def pick_pm(models):
    choices, weights = zip(*[(m,w) for m,w in models if w>0])
    return random.choices(choices, weights=weights)[0]

# ─────────────────────────────────────────────────────────────────────────────
# AZURE SKU CATALOG
# Each entry: (service, category, sku_description, usage_unit, unit_price,
#              qty_min_day, qty_max_day, pm_profile)
# ─────────────────────────────────────────────────────────────────────────────
AZURE_SKUS = [
  # Virtual Machines
  ('Virtual Machines','Compute','D2s v3 Spot/On-Demand','1 Hour',0.096,24,96,PM_COMPUTE),
  ('Virtual Machines','Compute','D4s v3 Spot/On-Demand','1 Hour',0.192,24,96,PM_COMPUTE),
  ('Virtual Machines','Compute','D8s v3 Spot/On-Demand','1 Hour',0.384,12,72,PM_COMPUTE),
  ('Virtual Machines','Compute','E4s v3 (Memory Opt)','1 Hour',0.252,12,48,PM_COMPUTE),
  ('Virtual Machines','Compute','NC6s v3 (GPU)','1 Hour',3.060,4,24,PM_COMPUTE),
  # AKS
  ('Azure Kubernetes Service','Compute','Standard_D4s_v3 Node','1 Hour',0.192,72,240,PM_COMPUTE),
  ('Azure Kubernetes Service','Compute','Standard_D8s_v3 Node','1 Hour',0.384,48,168,PM_COMPUTE),
  # App Service
  ('Azure App Service','Compute','P2v3 Plan','1 Hour',0.194,24,48,PM_DATA),
  ('Azure App Service','Compute','P3v3 Plan','1 Hour',0.388,12,24,PM_DATA),
  # Functions
  ('Azure Functions','Serverless','Execution Units','1M Executions',0.20,1,50,PM_OTHER),
  ('Azure Functions','Serverless','Execution Time','GB-s',0.000016,10000,500000,PM_OTHER),
  # Container Instances
  ('Azure Container Instances','Compute','vCPU Duration','vCPU-s',0.0000135,3600,86400,PM_OTHER),
  ('Azure Container Instances','Compute','Memory Duration','GB-s',0.0000015,7200,172800,PM_OTHER),
  # Batch
  ('Azure Batch','Compute','Standard_D4s_v3','1 Hour',0.192,24,120,PM_COMPUTE),
  # Databricks
  ('Azure Databricks','Analytics','Jobs Compute DBUs','DBU',0.150,100,2000,PM_DATA),
  ('Azure Databricks','Analytics','All-Purpose Compute DBUs','DBU',0.550,50,500,PM_DATA),
  ('Azure Databricks','Analytics','SQL Compute DBUs','DBU',0.550,100,1000,PM_DATA),
  ('Azure Databricks','Analytics','DLT Core DBUs','DBU',0.200,50,400,PM_DATA),
  # Synapse
  ('Azure Synapse Analytics','Analytics','Dedicated SQL Pool DWUs','DWH-Hour',1.200,24,48,PM_DATA),
  ('Azure Synapse Analytics','Analytics','Serverless SQL Queries','TB Processed',5.00,0.1,10,PM_OTHER),
  # HDInsight
  ('Azure HDInsight','Analytics','D4 v2 Worker Node','1 Hour',0.293,48,144,PM_COMPUTE),
  # Stream Analytics
  ('Azure Stream Analytics','Analytics','Streaming Units','SU/Hour',0.080,6,24,PM_OTHER),
  # Data Explorer
  ('Azure Data Explorer','Analytics','D11 v2 Compute','1 Hour',0.359,8,24,PM_COMPUTE),
  # Blob Storage
  ('Azure Blob Storage','Storage','LRS Data Stored','GB/Month',0.018,500,20000,PM_OTHER),
  ('Azure Blob Storage','Storage','GRS Data Stored','GB/Month',0.035,100,5000,PM_OTHER),
  ('Azure Blob Storage','Storage','Read Operations','10K',0.004,100,5000,PM_OTHER),
  ('Azure Blob Storage','Storage','Write Operations','10K',0.050,50,2000,PM_OTHER),
  # Data Lake Storage
  ('Azure Data Lake Storage','Storage','LRS Hierarchical Namespace','GB/Month',0.023,1000,50000,PM_OTHER),
  ('Azure Data Lake Storage','Storage','Read Transactions','10K',0.004,500,10000,PM_OTHER),
  # Files
  ('Azure Files','Storage','LRS File Storage','GB/Month',0.060,100,2000,PM_OTHER),
  # Backup
  ('Azure Backup','Storage','LRS Backup Storage','GB/Month',0.024,500,10000,PM_OTHER),
  # SQL Database
  ('Azure SQL Database','Database','General Purpose 4 vCores','1 Hour',0.725,24,24,PM_DATA),
  ('Azure SQL Database','Database','Business Critical 8 vCores','1 Hour',2.899,24,24,PM_DATA),
  ('Azure SQL Database','Database','Serverless 1-4 vCores','vCore-Hour',0.181,2,96,PM_OTHER),
  # Cosmos DB
  ('Azure Cosmos DB','Database','400 RU/s Provisioned','100 RU/s/Hour',0.008,24,24,PM_DATA),
  ('Azure Cosmos DB','Database','Transactional Storage','GB/Month',0.250,100,5000,PM_OTHER),
  # PostgreSQL
  ('Azure Database for PostgreSQL','Database','General Purpose 4 vCores','1 Hour',0.302,24,24,PM_DATA),
  # Redis
  ('Azure Cache for Redis','Database','C2 Standard','1 Hour',0.093,24,24,PM_DATA),
  ('Azure Cache for Redis','Database','P1 Premium','1 Hour',0.554,24,24,PM_DATA),
  # Data Factory
  ('Azure Data Factory','Integration','Data Flow vCore-Hour','vCore-Hour',0.274,10,200,PM_OTHER),
  ('Azure Data Factory','Integration','Orchestration Activity Runs','1K Runs',1.00,1,50,PM_OTHER),
  ('Azure Data Factory','Integration','Data Movement','DIU-Hour',0.250,5,100,PM_OTHER),
  # Event Hubs
  ('Azure Event Hubs','Integration','Standard Throughput Units','TU-Hour',0.030,8,32,PM_OTHER),
  ('Azure Event Hubs','Integration','Ingress Events','1M Events',0.028,10,500,PM_OTHER),
  # Service Bus
  ('Azure Service Bus','Integration','Standard Operations','1M Operations',0.10,1,50,PM_OTHER),
  # Logic Apps
  ('Azure Logic Apps','Integration','Action Executions','10K Actions',0.25,1,20,PM_OTHER),
  # API Management
  ('Azure API Management','Integration','Developer Tier Calls','1M Calls',3.50,0.1,5,PM_OTHER),
  ('Azure API Management','Integration','Standard Tier Calls','1M Calls',3.50,1,50,PM_OTHER),
  # OpenAI
  ('Azure OpenAI Service','AI / ML','GPT-4 Input Tokens','1K Tokens',0.03,100,10000,PM_OTHER),
  ('Azure OpenAI Service','AI / ML','GPT-4 Output Tokens','1K Tokens',0.06,50,5000,PM_OTHER),
  ('Azure OpenAI Service','AI / ML','Embeddings Tokens','1K Tokens',0.0001,1000,100000,PM_OTHER),
  # Machine Learning
  ('Azure Machine Learning','AI / ML','Compute Cluster NC6s v3','1 Hour',3.060,4,24,PM_COMPUTE),
  ('Azure Machine Learning','AI / ML','Managed Online Endpoint','1 Hour',0.096,24,48,PM_OTHER),
  # Cognitive Services
  ('Azure Cognitive Services','AI / ML','Language Transactions','1K Transactions',1.50,1,20,PM_OTHER),
  ('Azure Cognitive Services','AI / ML','Vision Transactions','1K Transactions',1.00,1,30,PM_OTHER),
  # Networking
  ('Azure Virtual Network','Networking','Peering Data Transfer','GB',0.010,100,10000,PM_OTHER),
  ('Azure Application Gateway','Networking','WAF v2 Gateway Hours','1 Hour',0.360,24,24,PM_OTHER),
  ('Azure Application Gateway','Networking','WAF v2 Capacity Units','CU-Hour',0.008,50,200,PM_OTHER),
  ('Azure Load Balancer','Networking','Standard LB Hours','1 Hour',0.005,24,24,PM_OTHER),
  ('Azure CDN','Networking','Zone 1 Data Transfer','GB',0.087,10,1000,PM_OTHER),
  ('Azure DNS','Networking','Hosted Zone','Month',0.50,1,20,PM_OTHER),
  ('Azure ExpressRoute','Networking','Standard Circuit Port','Month',55.0,1,4,PM_OTHER),
  ('Azure ExpressRoute','Networking','Metered Data','GB',0.025,100,10000,PM_OTHER),
  # Security
  ('Azure Key Vault','Security','Secrets Operations','10K Operations',0.03,1,100,PM_OTHER),
  ('Microsoft Defender for Cloud','Security','Server Plan 2','Node/Hour',0.017,50,200,PM_OTHER),
  ('Azure Firewall','Security','Deployment Hours','1 Hour',1.25,24,24,PM_OTHER),
  ('Azure Firewall','Security','Data Processed','GB',0.016,100,5000,PM_OTHER),
  # Management
  ('Azure Monitor','Management','Log Analytics Data Ingestion','GB',2.76,10,500,PM_OTHER),
  ('Azure Monitor','Management','Metrics API Calls','1M Calls',0.01,1,50,PM_OTHER),
  ('Azure Log Analytics','Management','Data Retention','GB/Month',0.10,100,5000,PM_OTHER),
  # DevOps
  ('Azure DevOps','DevOps','Basic + Test Plans User','User/Month',52.0,10,100,PM_OTHER),
  ('Azure DevOps','DevOps','Parallel Jobs','Job/Month',40.0,2,10,PM_OTHER),
  ('Azure Container Registry','DevOps','Standard Registry','Month',20.0,1,3,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# AWS SKU CATALOG
# ─────────────────────────────────────────────────────────────────────────────
AWS_SKUS = [
  # EC2
  ('Amazon EC2','Compute','BoxUsage:m5.xlarge','Hrs',0.192,24,96,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:m5.2xlarge','Hrs',0.384,12,72,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:r5.2xlarge','Hrs',0.504,12,48,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:c5.4xlarge','Hrs',0.680,8,48,PM_COMPUTE),
  ('Amazon EC2','Compute','SpotUsage:m5.xlarge','Hrs',0.058,24,96,PM_COMPUTE),
  ('Amazon EC2','Compute','BoxUsage:p3.2xlarge (GPU)','Hrs',3.060,4,24,PM_COMPUTE),
  # EKS
  ('Amazon EKS','Compute','AmazonEKS-Hours','Hrs',0.100,24,24,PM_OTHER),
  ('Amazon EKS','Compute','BoxUsage:m5.xlarge (Node)','Hrs',0.192,48,192,PM_COMPUTE),
  # ECS/Fargate
  ('Amazon ECS','Compute','BoxUsage:m5.large','Hrs',0.096,24,96,PM_COMPUTE),
  ('AWS Fargate','Serverless','Fargate-vCPU-Hours','vCPU-Hours',0.04048,10,200,PM_OTHER),
  ('AWS Fargate','Serverless','Fargate-GB-Hours','GB-Hours',0.004445,20,400,PM_OTHER),
  # Lambda
  ('AWS Lambda','Serverless','Lambda-GB-Second','GB-s',0.0000166667,10000,5000000,PM_OTHER),
  ('AWS Lambda','Serverless','Lambda-Request','1M Requests',0.20,1,100,PM_OTHER),
  # Batch
  ('AWS Batch','Compute','BoxUsage:c5.2xlarge','Hrs',0.340,8,48,PM_COMPUTE),
  # S3
  ('Amazon S3','Storage','TimedStorage-ByteHrs','GB-Mo',0.023,1000,50000,PM_OTHER),
  ('Amazon S3','Storage','Requests-Tier1 (PUT/COPY)','1K Requests',0.005,10,1000,PM_OTHER),
  ('Amazon S3','Storage','Requests-Tier2 (GET)','1K Requests',0.0004,100,10000,PM_OTHER),
  ('Amazon S3','Storage','DataTransfer-Out-Bytes','GB',0.090,10,1000,PM_OTHER),
  # EBS
  ('Amazon EBS','Storage','EBS:VolumeUsage.gp3','GB-Mo',0.080,500,20000,PM_OTHER),
  ('Amazon EBS','Storage','EBS:VolumeUsage.io2','GB-Mo',0.125,100,5000,PM_OTHER),
  # EFS
  ('Amazon EFS','Storage','EFS TimedStorage-ByteHrs','GB-Mo',0.30,100,5000,PM_OTHER),
  # Glacier
  ('Amazon S3 Glacier','Storage','Glacier TimedStorage','GB-Mo',0.004,500,20000,PM_OTHER),
  # FSx
  ('Amazon FSx','Storage','FSx:StorageUsage','GB-Mo',0.13,100,2000,PM_OTHER),
  # RDS
  ('Amazon RDS','Database','RDS:db.r5.large Multi-AZ','Hrs',0.48,24,24,PM_DATA),
  ('Amazon RDS','Database','RDS:db.r5.2xlarge Multi-AZ','Hrs',0.96,24,24,PM_DATA),
  ('Amazon RDS','Database','RDS:StorageUsage','GB-Mo',0.115,500,10000,PM_OTHER),
  # Aurora
  ('Amazon Aurora','Database','Aurora:ServerlessV2Usage','ACU-Hr',0.12,10,500,PM_DATA),
  ('Amazon Aurora','Database','Aurora:db.r5.2xlarge','Hrs',0.58,24,24,PM_DATA),
  ('Amazon Aurora','Database','Aurora:StorageUsage','GB-Mo',0.10,200,10000,PM_OTHER),
  # Redshift
  ('Amazon Redshift','Analytics','NodeUsage:dc2.8xlarge','Hrs',4.80,24,24,PM_DATA),
  ('Amazon Redshift','Analytics','NodeUsage:ra3.4xlarge','Hrs',3.26,24,48,PM_DATA),
  ('Amazon Redshift','Analytics','Serverless RPU-Hour','RPU-Hr',0.375,2,64,PM_OTHER),
  # DynamoDB
  ('Amazon DynamoDB','Database','DDB:WriteCapacityUnit-Hrs','WCU-Hr',0.00065,1000,50000,PM_OTHER),
  ('Amazon DynamoDB','Database','DDB:ReadCapacityUnit-Hrs','RCU-Hr',0.00013,5000,200000,PM_OTHER),
  ('Amazon DynamoDB','Database','TimedStorage-ByteHrs','GB-Mo',0.25,50,2000,PM_OTHER),
  # ElastiCache
  ('Amazon ElastiCache','Database','NodeUsage:cache.r6g.large','Hrs',0.166,24,48,PM_DATA),
  ('Amazon ElastiCache','Database','NodeUsage:cache.r6g.2xlarge','Hrs',0.665,24,24,PM_DATA),
  # DocumentDB
  ('Amazon DocumentDB','Database','InstanceUsage:db.r5.xlarge','Hrs',0.277,24,24,PM_DATA),
  # Neptune
  ('Amazon Neptune','Database','InstanceUsage:db.r5.xlarge','Hrs',0.295,24,24,PM_DATA),
  # Timestream
  ('Amazon Timestream','Database','TimestreamWrite-Records','1M Records',0.50,1,50,PM_OTHER),
  ('Amazon Timestream','Database','TimestreamStorage-Memory','GB-Hr',0.036,10,100,PM_OTHER),
  # EMR
  ('Amazon EMR','Analytics','EMR:m5.2xlarge','Hrs',0.096,48,192,PM_COMPUTE),
  ('Amazon EMR','Analytics','EMR:r5.4xlarge','Hrs',0.192,24,96,PM_COMPUTE),
  # Glue
  ('AWS Glue','Analytics','Glue-DPU-Hour (ETL)','DPU-Hr',0.44,2,100,PM_OTHER),
  ('AWS Glue','Analytics','Glue-DPU-Hour (Crawler)','DPU-Hr',0.44,1,20,PM_OTHER),
  # Athena
  ('Amazon Athena','Analytics','DataScannedInTB','TB',5.00,0.1,10,PM_OTHER),
  # QuickSight
  ('Amazon QuickSight','Analytics','QuickSight-User-Month','User-Mo',18.0,10,100,PM_OTHER),
  # SageMaker
  ('Amazon SageMaker','AI / ML','Training:ml.p3.2xlarge','Hrs',3.825,4,24,PM_COMPUTE),
  ('Amazon SageMaker','AI / ML','Endpoint:ml.c5.2xlarge','Hrs',0.454,24,48,PM_OTHER),
  ('Amazon SageMaker','AI / ML','Studio:ml.t3.medium','Hrs',0.046,8,24,PM_OTHER),
  # Bedrock
  ('Amazon Bedrock','AI / ML','Claude 3 Sonnet Input','1K Tokens',0.003,100,10000,PM_OTHER),
  ('Amazon Bedrock','AI / ML','Claude 3 Sonnet Output','1K Tokens',0.015,50,5000,PM_OTHER),
  ('Amazon Bedrock','AI / ML','Titan Embeddings Tokens','1K Tokens',0.0001,1000,100000,PM_OTHER),
  # Comprehend / Rekognition / Textract
  ('Amazon Comprehend','AI / ML','Comprehend-Units','Units',0.0001,1000,100000,PM_OTHER),
  ('Amazon Rekognition','AI / ML','Rekognition-Images','1K Images',1.00,1,50,PM_OTHER),
  ('Amazon Textract','AI / ML','Textract-Pages','1K Pages',1.50,0.1,10,PM_OTHER),
  ('Amazon Forecast','AI / ML','Forecast-DataPoints','1K Points',0.60,1,50,PM_OTHER),
  # Kinesis
  ('Amazon Kinesis','Integration','ShardHour','Shard-Hr',0.015,24,240,PM_OTHER),
  ('Amazon Kinesis','Integration','PUT-Payload-Unit','1M Units',0.014,100,10000,PM_OTHER),
  # MSK
  ('Amazon MSK','Integration','MSK:kafka.m5.large','Hrs',0.216,24,48,PM_DATA),
  # SNS/SQS
  ('Amazon SNS','Integration','SNS-Requests','1M Requests',0.50,0.1,10,PM_OTHER),
  ('Amazon SQS','Integration','SQS-Requests','1M Requests',0.40,1,50,PM_OTHER),
  # API Gateway
  ('Amazon API Gateway','Integration','REST API Calls','1M Calls',3.50,0.5,50,PM_OTHER),
  ('Amazon API Gateway','Integration','WebSocket Messages','1M Messages',1.00,1,100,PM_OTHER),
  # Step Functions
  ('AWS Step Functions','Integration','StateTransition','1K Transitions',0.025,10,1000,PM_OTHER),
  # Networking
  ('Amazon CloudFront','Networking','DataTransfer-Out-Bytes','GB',0.085,100,5000,PM_OTHER),
  ('Amazon CloudFront','Networking','Requests-HTTPS','10K Requests',0.010,10,1000,PM_OTHER),
  ('Amazon VPC','Networking','NatGateway-Hours','Hrs',0.045,24,24,PM_OTHER),
  ('Amazon VPC','Networking','NatGateway-Bytes','GB',0.045,10,1000,PM_OTHER),
  ('Amazon Route 53','Networking','HostedZone','Month',0.50,5,30,PM_OTHER),
  ('AWS Direct Connect','Networking','DataXfer-Out','GB',0.020,100,10000,PM_OTHER),
  ('Elastic Load Balancing','Networking','LoadBalancerUsage','Hrs',0.008,24,24,PM_OTHER),
  ('Elastic Load Balancing','Networking','LCUUsage','LCU-Hrs',0.008,10,200,PM_OTHER),
  ('AWS Data Transfer','Networking','DataTransfer-Regional-Bytes','GB',0.010,100,5000,PM_OTHER),
  # Security
  ('AWS KMS','Security','KMS-Requests','10K Requests',0.03,1,100,PM_OTHER),
  ('AWS WAF','Security','WebACL-Month','Month',5.00,1,5,PM_OTHER),
  ('Amazon Cognito','Security','MAU','MAU',0.0055,1000,50000,PM_OTHER),
  ('AWS Secrets Manager','Security','Secret-Month','Secret-Mo',0.40,10,200,PM_OTHER),
  ('AWS Shield','Security','Shield Advanced','Month',3000,1,1,PM_OTHER),
  # Management
  ('Amazon CloudWatch','Management','CW:MetricMonitorUsage','Metric-Mo',0.30,10,500,PM_OTHER),
  ('Amazon CloudWatch','Management','CW:LogsStorage','GB-Mo',0.03,50,2000,PM_OTHER),
  ('AWS CloudTrail','Management','CloudTrail-Event','100K Events',0.10,1,100,PM_OTHER),
  ('AWS Systems Manager','Management','SSM-AdvancedInstanceHour','Adv.Inst-Hr',0.00695,24,240,PM_OTHER),
  ('AWS Backup','Management','BackupStorage-AmazonS3','GB-Mo',0.05,500,10000,PM_OTHER),
  # DevOps
  ('AWS CodePipeline','DevOps','CodePipeline-Pipeline','Pipeline-Mo',1.00,5,30,PM_OTHER),
  ('AWS CodeBuild','DevOps','CodeBuild-Minute','Build-Min',0.005,100,5000,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# GCP SKU CATALOG
# ─────────────────────────────────────────────────────────────────────────────
GCP_SKUS = [
  # Compute Engine
  ('Compute Engine','Compute','N2 Instance Core running in Americas','vCPU-Hour',0.031611,24,240,PM_COMPUTE),
  ('Compute Engine','Compute','N2 Instance Ram running in Americas','GB-Hour',0.004237,48,480,PM_COMPUTE),
  ('Compute Engine','Compute','N2D AMD Instance Core running in Americas','vCPU-Hour',0.027027,24,192,PM_COMPUTE),
  ('Compute Engine','Compute','Spot Preemptible N2 Instance Core','vCPU-Hour',0.007897,24,240,PM_COMPUTE),
  ('Compute Engine','Compute','Nvidia Tesla T4 GPU attached to Spot','GPU-Hour',0.130,4,24,PM_COMPUTE),
  ('Compute Engine','Compute','Nvidia A100 80GB GPU','GPU-Hour',2.933,2,16,PM_COMPUTE),
  ('Compute Engine','Compute','Storage PD Capacity','GB-Month',0.040,500,20000,PM_OTHER),
  # GKE
  ('Google Kubernetes Engine','Compute','Zonal Cluster Management Fee','Hour',0.10,24,24,PM_OTHER),
  ('Google Kubernetes Engine','Compute','N2 Standard 4 Node','vCPU-Hour',0.031611,96,384,PM_COMPUTE),
  # Cloud Run
  ('Cloud Run','Serverless','CPU Allocation Time','vCPU-Second',0.000024,1000,100000,PM_OTHER),
  ('Cloud Run','Serverless','Memory Allocation Time','GB-Second',0.0000025,2000,200000,PM_OTHER),
  ('Cloud Run','Serverless','Requests','1M Requests',0.40,1,100,PM_OTHER),
  # Cloud Functions
  ('Cloud Functions','Serverless','Invocations','1M Invocations',0.40,0.1,20,PM_OTHER),
  ('Cloud Functions','Serverless','Compute Time','GB-Second',0.0000025,5000,500000,PM_OTHER),
  # App Engine
  ('App Engine','Compute','B4 Instance Hours','Instance-Hour',0.05,24,96,PM_OTHER),
  ('App Engine','Compute','F4 Frontend Instance Hours','Instance-Hour',0.05,8,32,PM_OTHER),
  # Batch
  ('Batch','Compute','N2 vCPU','vCPU-Hour',0.031611,10,200,PM_COMPUTE),
  # Cloud Storage
  ('Cloud Storage','Storage','Standard Storage','GB-Month',0.020,1000,50000,PM_OTHER),
  ('Cloud Storage','Storage','Nearline Storage','GB-Month',0.010,500,20000,PM_OTHER),
  ('Cloud Storage','Storage','Class A Operations','10K Ops',0.05,10,1000,PM_OTHER),
  ('Cloud Storage','Storage','Network Egress','GB',0.12,10,1000,PM_OTHER),
  # Persistent Disk
  ('Persistent Disk','Storage','SSD backed PD Capacity','GB-Month',0.170,200,5000,PM_OTHER),
  ('Persistent Disk','Storage','Standard PD Capacity','GB-Month',0.040,500,20000,PM_OTHER),
  # Filestore
  ('Filestore','Storage','Basic HDD Capacity','GB-Month',0.200,500,5000,PM_OTHER),
  # Cloud Backup
  ('Cloud Backup','Storage','Backup Storage','GB-Month',0.023,200,10000,PM_OTHER),
  # Cloud SQL
  ('Cloud SQL','Database','DB n1-standard-4 (MySQL)','Hour',0.385,24,24,PM_DATA),
  ('Cloud SQL','Database','DB n1-highmem-8 (PostgreSQL)','Hour',0.754,24,24,PM_DATA),
  ('Cloud SQL','Database','High Availability','Hour',0.770,24,24,PM_DATA),
  ('Cloud SQL','Database','Storage Capacity SSD','GB-Month',0.170,200,5000,PM_OTHER),
  # Cloud Spanner
  ('Cloud Spanner','Database','Processing Unit','Node-Hour',0.90,24,240,PM_DATA),
  ('Cloud Spanner','Database','Storage','GB-Month',0.300,100,5000,PM_OTHER),
  # Bigtable
  ('Cloud Bigtable','Database','SSD Storage Node','Node-Hour',0.65,24,48,PM_DATA),
  ('Cloud Bigtable','Database','SSD Storage','GB-Month',0.17,100,5000,PM_OTHER),
  # Firestore
  ('Firestore','Database','Document Reads','100K Reads',0.06,10,1000,PM_OTHER),
  ('Firestore','Database','Document Writes','100K Writes',0.18,5,500,PM_OTHER),
  ('Firestore','Database','Storage','GB-Month',0.18,10,500,PM_OTHER),
  # Memorystore
  ('Memorystore','Database','Redis Basic M1','GB-Hour',0.049,24,24,PM_DATA),
  ('Memorystore','Database','Redis Standard M2','GB-Hour',0.098,24,24,PM_DATA),
  # AlloyDB
  ('AlloyDB','Database','vCPU','vCPU-Hour',0.0715,24,96,PM_DATA),
  ('AlloyDB','Database','Memory','GB-Hour',0.00875,96,384,PM_DATA),
  # BigQuery
  ('BigQuery','Analytics','Analysis','TiB',6.25,0.1,20,PM_OTHER),
  ('BigQuery','Analytics','Active Logical Storage','GB-Month',0.020,1000,100000,PM_OTHER),
  ('BigQuery','Analytics','Long Term Storage','GB-Month',0.010,5000,500000,PM_OTHER),
  ('BigQuery','Analytics','BI Engine Reservation','GB-Month',8.00,10,100,PM_OTHER),
  # Dataproc
  ('Dataproc','Analytics','Dataproc Premium n1-standard-4','vCPU-Hour',0.01,24,120,PM_COMPUTE),
  ('Dataproc','Analytics','Dataproc Preemptible n1-standard-4','vCPU-Hour',0.005,24,96,PM_COMPUTE),
  # Dataflow
  ('Dataflow','Analytics','Dataflow Shuffle','GB',0.011,100,10000,PM_OTHER),
  ('Dataflow','Analytics','vCPU Running','vCPU-Hour',0.056,10,200,PM_OTHER),
  # Looker
  ('Looker','Analytics','Developer User License','User-Month',125.0,5,20,PM_OTHER),
  ('Looker','Analytics','Standard User License','User-Month',30.0,20,100,PM_OTHER),
  # Data Fusion
  ('Cloud Data Fusion','Analytics','Basic Edition','Instance-Hour',0.35,24,24,PM_OTHER),
  # Vertex AI
  ('Vertex AI','AI / ML','Training: n1-standard-8','Node-Hour',0.380,4,48,PM_COMPUTE),
  ('Vertex AI','AI / ML','Training: a2-highgpu-1g (A100)','Node-Hour',3.673,2,16,PM_COMPUTE),
  ('Vertex AI','AI / ML','Prediction: n1-standard-4','Node-Hour',0.190,24,48,PM_OTHER),
  ('Vertex AI','AI / ML','Gemini 1.5 Pro Input Tokens','1K Tokens',0.00125,500,50000,PM_OTHER),
  ('Vertex AI','AI / ML','Gemini 1.5 Pro Output Tokens','1K Tokens',0.005,100,10000,PM_OTHER),
  ('Vertex AI','AI / ML','Model Garden Inference','1K Predictions',0.10,10,1000,PM_OTHER),
  # AI APIs
  ('Natural Language AI','AI / ML','Natural Language Units','1K Units',1.00,1,50,PM_OTHER),
  ('Vision AI','AI / ML','Label Detection Images','1K Images',1.50,0.5,20,PM_OTHER),
  ('Speech-to-Text','AI / ML','Speech Recognition','1 Minute',0.016,100,5000,PM_OTHER),
  ('Document AI','AI / ML','Document Pages','1K Pages',1.50,0.1,10,PM_OTHER),
  ('Recommendations AI','AI / ML','Prediction Requests','1K Requests',0.27,10,1000,PM_OTHER),
  # Pub/Sub
  ('Pub/Sub','Integration','Message Delivery','TiB',60.0,0.001,0.5,PM_OTHER),
  # Cloud Composer
  ('Cloud Composer','Integration','Composer 2 vCPU','vCPU-Hour',0.062,24,120,PM_OTHER),
  # Apigee
  ('Apigee API Management','Integration','API Calls','1M Calls',3.50,0.5,50,PM_OTHER),
  ('Apigee API Management','Integration','Environment Unit','Hour',0.17,24,24,PM_OTHER),
  # Eventarc / Tasks
  ('Eventarc','Integration','Events','1M Events',0.10,1,50,PM_OTHER),
  ('Cloud Tasks','Integration','Task Operations','1M Ops',0.40,0.5,20,PM_OTHER),
  # Networking
  ('Cloud Networking','Networking','Premium Tier Egress','GB',0.12,10,1000,PM_OTHER),
  ('Cloud Networking','Networking','Standard Tier Egress','GB',0.085,5,500,PM_OTHER),
  ('Cloud Load Balancing','Networking','Forwarding Rule Charge','Hour',0.025,24,24,PM_OTHER),
  ('Cloud Load Balancing','Networking','Data Processed','GB',0.008,100,5000,PM_OTHER),
  ('Cloud CDN','Networking','Cache Egress','GB',0.08,10,500,PM_OTHER),
  ('Cloud DNS','Networking','Managed Zone','Zone-Month',0.20,5,20,PM_OTHER),
  ('Cloud Interconnect','Networking','Dedicated 10Gbps Port','Month',1700.0,1,2,PM_OTHER),
  # Security
  ('Cloud Armor','Security','WAF Rule Evaluation','1M Requests',0.75,1,100,PM_OTHER),
  ('Cloud KMS','Security','Key Versions','Key-Month',0.06,20,200,PM_OTHER),
  ('Secret Manager','Security','Active Secret Versions','Month',0.06,20,200,PM_OTHER),
  ('Cloud Identity','Security','Cloud Identity Premium','User-Month',6.00,50,500,PM_OTHER),
  # Monitoring / DevOps
  ('Cloud Monitoring','Management','Monitoring Data Ingestion','MiB',0.258,100,5000,PM_OTHER),
  ('Cloud Logging','Management','Log Volume','GiB',0.50,10,500,PM_OTHER),
  ('Artifact Registry','DevOps','Storage','GB-Month',0.10,50,500,PM_OTHER),
  ('Cloud Build','DevOps','Build Minutes','Build-Min',0.003,100,5000,PM_OTHER),
  ('Firebase','Mobile/Web','Spark Plan Overages','GB',0.026,5,100,PM_OTHER),
]

# ─────────────────────────────────────────────────────────────────────────────
# Row generator
# ─────────────────────────────────────────────────────────────────────────────
NOW = datetime(2026, 4, 26)
NINETY_AGO = NOW - timedelta(days=89)
random.seed(9999)

def gen_rows(cloud, accounts, skus, start, end):
    rows = []
    cur = start
    while cur <= end:
        dow  = cur.weekday()
        wf   = 0.55 if dow >= 5 else 1.0
        mf   = 1.0 + 0.15 * (cur.day / 28)
        ds   = cur.strftime('%Y-%m-%d')
        ts   = cur.strftime('%Y-%m-%d %H:%M:%S')
        for acct_id, acct_name, region in accounts:
            rmul = REGION_MUL.get(region, 1.0)
            for svc, cat, sku_desc, unit, base_price, qmin, qmax, pm_def in skus:
                pm      = pick_pm(pm_def)
                disc    = RESERVED_DISC[pm]
                unit_p  = round(base_price * rmul * disc, 8)
                qty     = round(random.uniform(qmin, qmax) * wf * mf, 4)
                cost    = round(unit_p * qty * random.uniform(0.92, 1.08), 4)
                tags    = make_tags()
                rg      = f'rg-{acct_name.lower().replace(" ","-")}' if cloud == 'azure' else ''
                rows.append((cloud, acct_id, acct_name, ds, svc, cat,
                              sku_desc, unit, qty, unit_p, cost, 'USD',
                              pm, region, rg, tags, ts))
        cur += timedelta(days=1)
    return rows

azure_rows = gen_rows('azure', AZURE_SUBSCRIPTIONS, AZURE_SKUS, NINETY_AGO, NOW)
aws_rows   = gen_rows('aws',   AWS_ACCOUNTS,        AWS_SKUS,   NINETY_AGO, NOW)
gcp_rows   = gen_rows('gcp',   GCP_PROJECTS,        GCP_SKUS,   NINETY_AGO, NOW)
all_rows   = azure_rows + aws_rows + gcp_rows

print(f'Azure  : {len(azure_rows):>7,} rows  ({len(AZURE_SKUS)} SKUs × {len(AZURE_SUBSCRIPTIONS)} subscriptions × 90d)')
print(f'AWS    : {len(aws_rows):>7,} rows  ({len(AWS_SKUS)} SKUs × {len(AWS_ACCOUNTS)} accounts × 90d)')
print(f'GCP    : {len(gcp_rows):>7,} rows  ({len(GCP_SKUS)} SKUs × {len(GCP_PROJECTS)} projects × 90d)')
print(f'Total  : {len(all_rows):>7,} rows')
az_cost  = sum(r[10] for r in azure_rows)
aws_cost = sum(r[10] for r in aws_rows)
gcp_cost = sum(r[10] for r in gcp_rows)
print(f'\nEst. 90-day spend:')
print(f'  Azure  : ${az_cost:>12,.0f}')
print(f'  AWS    : ${aws_cost:>12,.0f}')
print(f'  GCP    : ${gcp_cost:>12,.0f}')
print(f'  Total  : ${az_cost+aws_cost+gcp_cost:>12,.0f}')


In [ ]:
CATALOG = 'workspace'
TARGET  = f'{CATALOG}.platform.cloud_platform_costs'

run_sql('create platform schema', f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.platform')
run_sql('drop cloud costs',       f'DROP TABLE IF EXISTS {TARGET}')
run_sql('create cloud costs', f'''
CREATE TABLE {TARGET} (
    cloud_provider   STRING   NOT NULL COMMENT 'azure | aws | gcp',
    account_id       STRING   NOT NULL COMMENT 'Subscription ID / AWS Account ID / GCP Project ID',
    account_name     STRING            COMMENT 'Human-readable account name',
    usage_date       DATE     NOT NULL,
    service_name     STRING            COMMENT 'Top-level cloud service (e.g. Amazon EC2)',
    service_category STRING            COMMENT 'Compute | Storage | Database | Analytics | AI / ML | ...',
    sku_description  STRING            COMMENT 'Meter/SKU level (e.g. BoxUsage:m5.xlarge)',
    usage_unit       STRING            COMMENT 'Unit of measure (Hrs, GB-Mo, 1K Tokens, ...)',
    usage_quantity   DOUBLE            COMMENT 'Quantity consumed in this unit',
    unit_price       DOUBLE            COMMENT 'Price per unit (after reserved/spot discount)',
    cost_usd         DOUBLE            COMMENT 'usage_quantity * unit_price',
    currency         STRING,
    pricing_model    STRING            COMMENT 'ON_DEMAND | RESERVED_1Y | RESERVED_3Y | SPOT | COMMITTED_1Y',
    region           STRING,
    resource_group   STRING            COMMENT 'Azure resource group; empty for AWS/GCP',
    tags             MAP<STRING, STRING> COMMENT 'team, environment, cost_center, project',
    ingested_at      TIMESTAMP
)
USING DELTA
PARTITIONED BY (cloud_provider, usage_date)
TBLPROPERTIES (\'delta.autoOptimize.optimizeWrite\' = \'true\',
               \'delta.autoOptimize.autoCompact\'   = \'true\')
''')
print('Table created:', TARGET)

# Build SQL value strings
def sql_esc(s): return str(s).replace("'", "''")

sql_rows = []
for r in all_rows:
    cloud, acct_id, acct_name, ds, svc, cat, sku, unit, qty, uprice, cost, curr, pm, region, rg, tags_json, ts = r
    # Parse tags JSON → SQL map literal
    import json as _j
    t = _j.loads(tags_json)
    tags_sql = "map(" + ",".join(f"'{k}','{sql_esc(v)}'" for k,v in t.items()) + ")"
    sql_rows.append(
        f"('{cloud}','{acct_id}','{sql_esc(acct_name)}','{ds}',"
        f"'{sql_esc(svc)}','{cat}',"
        f"'{sql_esc(sku)}','{unit}',{qty},{uprice},{cost},'{curr}',"
        f"'{pm}','{region}','{sql_esc(rg)}',"
        f"{tags_sql},'{ts}')"
    )

print(f'Inserting {len(sql_rows):,} rows in chunks of 300...')
for i, chunk in enumerate(chunk_list(sql_rows, 300)):
    run_sql(f'cloud cost chunk {i+1}/{(len(sql_rows)//300)+1}',
            f'INSERT INTO {TARGET} VALUES\n' + ',\n'.join(chunk))

print(f'\n✅ cloud_platform_costs loaded: {len(sql_rows):,} rows')
run_sql('verify',
    f"SELECT cloud_provider, pricing_model, COUNT(*) AS rows, "
    f"ROUND(SUM(cost_usd),0) AS total_usd "
    f"FROM {TARGET} GROUP BY 1,2 ORDER BY 1,2")
